# Random Forest Training (V2)
This notebook trains:
- one binary quality model (`is_good`: 1 good / 0 bad)
- one model per error reason (`err_*` columns)


In [1]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


In [2]:
DATASET_FILE = "reps_dataset_v2.csv"
MODEL_FILE = "bicep_curl_model.pkl"

FEATURE_COLUMNS = [
    "min_left_angle",
    "max_left_angle",
    "min_right_angle",
    "max_right_angle",
    "rep_duration",
    "left_rom",
    "right_rom",
    "elbow_rom_diff",
    "concentric_duration",
    "eccentric_duration",
    "left_peak_velocity",
    "right_peak_velocity",
    "torso_lean_mean",
    "torso_lean_max",
    "torso_sway",
    "left_elbow_drift",
    "right_elbow_drift",
    "pose_visibility_mean",
    "pose_visibility_min",
    "tracking_lost_ratio",
]

ERROR_COLUMNS = [
    "err_partial_rom",
    "err_too_fast",
    "err_torso_sway",
    "err_elbow_drift",
    "err_asymmetry",
    "err_shoulder_swing",
    "err_wrist_compensation",
    "err_control_loss",
]

TARGET_COLUMN = "is_good"


In [3]:
df = pd.read_csv(DATASET_FILE)
print("Dataset loaded:", DATASET_FILE)
print("Rows:", len(df))

required_columns = FEATURE_COLUMNS + ERROR_COLUMNS + [TARGET_COLUMN]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN]).copy()
df[ERROR_COLUMNS] = df[ERROR_COLUMNS].fillna(0).astype(int)
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

print("\nQuality label counts (is_good):")
print(df[TARGET_COLUMN].value_counts().sort_index())

print("\nReason label counts:")
for col in ERROR_COLUMNS:
    counts = df[col].value_counts().sort_index().to_dict()
    print(f"{col}: {counts}")

df.head()


Dataset loaded: reps_dataset_v2.csv
Rows: 49

Quality label counts (is_good):
is_good
0    25
1    24
Name: count, dtype: int64

Reason label counts:
err_partial_rom: {0: 42, 1: 7}
err_too_fast: {0: 41, 1: 8}
err_torso_sway: {0: 45, 1: 4}
err_elbow_drift: {0: 49}
err_asymmetry: {0: 43, 1: 6}
err_shoulder_swing: {0: 49}
err_wrist_compensation: {0: 49}
err_control_loss: {0: 49}


,min_left_angle,max_left_angle,min_right_angle,max_right_angle,rep_duration,left_rom,right_rom,elbow_rom_diff,concentric_duration,eccentric_duration,...,tracking_lost_ratio,err_partial_rom,err_too_fast,err_torso_sway,err_elbow_drift,err_asymmetry,err_shoulder_swing,err_wrist_compensation,err_control_loss,is_good
0,13.87,177.79,17.97,179.98,7.61,163.91,162.02,1.89,6.92,0.62,...,0.0,0,0,0,0,0,0,0,0,1
1,18.98,177.70,25.16,179.98,3.43,158.73,154.82,3.90,3.24,0.14,...,0.0,0,1,0,0,0,0,0,0,0
2,8.48,179.96,16.86,179.50,4.69,171.48,162.64,8.85,4.07,0.58,...,0.0,0,0,0,0,1,0,0,0,0
3,12.91,179.76,18.00,179.99,8.96,166.85,161.99,4.86,8.43,0.48,...,0.0,0,0,0,0,0,0,0,0,1
4,0.38,179.98,13.19,179.86,3.84,179.60,166.67,12.93,3.71,0.08,...,0.0,0,1,0,0,0,0,0,0,0


In [4]:
X = df[FEATURE_COLUMNS]
y_quality = df[TARGET_COLUMN]

can_train_quality = y_quality.nunique() >= 2
can_do_stratified_split = (
    can_train_quality
    and len(df) >= 10
    and y_quality.value_counts().min() >= 2
)

if can_do_stratified_split:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_quality,
        test_size=0.2,
        random_state=42,
        stratify=y_quality,
    )
    evaluate_quality = True
    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)
else:
    X_train, y_train = X, y_quality
    X_test, y_test = None, None
    evaluate_quality = False
    print("Not enough balanced data for a stratified test split.")
    print("Training quality model on full dataset and skipping test metrics.")


Train shape: (39, 20)
Test shape: (10, 20)


In [5]:
quality_model = None

if can_train_quality:
    quality_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
    )
    quality_model.fit(X_train, y_train)
    print("Quality model trained.")
else:
    print("Quality model not trained: only one class found in is_good.")


Quality model trained.


In [6]:
if quality_model is not None:
    train_accuracy = quality_model.score(X_train, y_train)
    print("Quality train accuracy:", round(train_accuracy, 4))

    if evaluate_quality:
        y_pred = quality_model.predict(X_test)
        print("Quality test accuracy:", round(accuracy_score(y_test, y_pred), 4))
        print("\nQuality classification report:")
        print(classification_report(y_test, y_pred, zero_division=0))
        print("Quality confusion matrix:")
        print(confusion_matrix(y_test, y_pred))
    else:
        print("Quality test metrics skipped (insufficient balanced data).")


Quality train accuracy: 1.0
Quality test accuracy: 0.7

Quality classification report:
              precision    recall  f1-score   support

           0       0.75      0.60      0.67         5
           1       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10

Quality confusion matrix:
[[3 2]
 [1 4]]


In [7]:
error_models = {}

for error_col in ERROR_COLUMNS:
    y_error = df[error_col]

    if y_error.nunique() < 2:
        error_models[error_col] = {
            "type": "constant",
            "value": int(y_error.iloc[0]),
        }
        print(f"{error_col}: only one class ({int(y_error.iloc[0])}), saved as constant predictor.")
        continue

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
    )
    model.fit(X, y_error)
    error_models[error_col] = {
        "type": "model",
        "model": model,
    }
    print(f"{error_col}: model trained.")


err_partial_rom: model trained.
err_too_fast: model trained.
err_torso_sway: model trained.
err_elbow_drift: only one class (0), saved as constant predictor.
err_asymmetry: model trained.
err_shoulder_swing: only one class (0), saved as constant predictor.
err_wrist_compensation: only one class (0), saved as constant predictor.
err_control_loss: only one class (0), saved as constant predictor.


In [8]:
bundle = {
    "version": "v2_binary_quality_plus_reasons",
    "dataset_file": DATASET_FILE,
    "feature_columns": FEATURE_COLUMNS,
    "error_columns": ERROR_COLUMNS,
    "target_column": TARGET_COLUMN,
    "quality_model": quality_model,
    "error_models": error_models,
}

joblib.dump(bundle, MODEL_FILE)
print(f"Saved model bundle to: {MODEL_FILE}")


Saved model bundle to: bicep_curl_model.pkl


## Notes
- If `is_good` has only one class (all 0 or all 1), binary quality model training is skipped.
- Reason models with only one class are stored as constant predictors.
